# Other Representations of Rotations

Source orientation: printed pages 575-584, PDF pages 593-602. This notebook is an original visualization-first lesson inspired by the structure of *Modern Robotics: Mechanics, Planning, and Control* by Kevin M. Lynch and Frank C. Park. It does not quote or reproduce textbook prose, exercises, screenshots, or page crops.

## Chapter Question

Why do so many rotation coordinates exist, and what does each one trade away?

This question is the thread for the chapter. The goal is not to memorize a list of formulas; it is to make the geometry inspectable. A robot mechanism is a physical machine, but the way we reason about it is through spaces, maps, constraints, tangents, and dual forces. The notebook keeps those objects visible. Every diagram and computation below is designed to answer a local question that a reader can check: what is moving, what is fixed, what map is being applied, what invariant should survive, and where can the model fail?

## Translation Guide

The source chapter or appendix is translated into computational language using these terms: Euler angles, roll-pitch-yaw, unit quaternion, Cayley-Rodrigues, singularity, double cover, minimal coordinate. The important conversions for this notebook are:

- Three-parameter coordinates are compact but cannot be globally nonsingular.
- Unit quaternions avoid gimbal lock by adding a constraint and a sign ambiguity.
- Cayley parameters exclude rotations by pi.

The central habit is to name the representation and the invariant at the same time. Coordinates are useful only when we know what geometric object they represent. In this appendix the invariant is always the same rotation in SO(3): a linear map that preserves lengths, preserves angles, and has determinant one. Euler angles and roll-pitch-yaw coordinates describe that map by composing three elementary rotations. Unit quaternions describe it as a point on the unit 3-sphere with a sign ambiguity. Cayley-Rodrigues parameters describe it by a rational chart that becomes unusable at rotations by pi. The notebook therefore pairs each formula with a small experiment: a plotted curve, an orthogonality residual, a determinant check, or a conversion that should leave the rotated vector unchanged.

## Representation Contracts

Every coordinate system in this appendix makes a different engineering contract with the same underlying object. A three-angle convention is human-readable and fits common user interfaces, but it depends on an ordered convention: changing from body-fixed to space-fixed rotations, or from ZYX to XYZ order, changes the meaning of the same three numbers. A unit quaternion uses four numbers for a three-dimensional attitude, so it trades minimality for a smooth global container; the price is that `q` and `-q` are the same physical rotation. Cayley parameters are compact and often pleasant in algebra because they avoid trigonometric functions after the parameter is chosen, but the norm grows like `tan(theta / 2)` and therefore warns us before a pi rotation leaves the chart.

The computational test is not whether the coordinates look familiar. The test is whether the reconstructed rotation matrix is orthonormal, whether its determinant stays near one, and whether two supposedly equivalent coordinates rotate every vector the same way. When those checks pass, the representation is serving the geometry. When they fail, the error usually lives in a convention mismatch, a missing normalization, a sign discontinuity, or an attempt to use a local chart outside its neighborhood.

## Route Through the Notebook

1. Establish the objects and conventions needed for other representations of rotations.
2. Build a concept dependency map so definitions have visible structure.
3. Generate the main visual lab: Euler angles, roll-pitch-yaw, quaternions, and Cayley parameters.
4. Run a compact worked example that exposes an invariant.
5. Try an applied lab that changes a parameter and asks what should remain true.
6. Finish with sanity checks that assert artifact existence, image variation, and numerical margins.

This is a standalone course notebook. The PDF can be useful for historical context and exercises, but the lesson here is complete without it. Definitions are restated in fresh language, examples are computed from scratch, and visuals are generated by course-local code. When the notebook mentions a source span, it is a navigation reference rather than a dependency for comprehension.

## Visualization Storyboard

| Concept | Representation | Artifact | Inspection target |
| --- | --- | --- | --- |
| concept dependency map | directed graph | `artifacts/appendix-b-other-representations-of-rotations/figures/concept-dependency-map.png` | which definitions feed the chapter's computation |
| Euler angles, roll-pitch-yaw, quaternions, and Cayley parameters | rotation parameter comparison curves | `artifacts/appendix-b-other-representations-of-rotations/figures/rotation-representations-lab.png` | where each parameterization is smooth, singular, or redundant |
| numeric invariant check | residual bar chart and JSON summary | `artifacts/appendix-b-other-representations-of-rotations/figures/rotation-representations-checks.png` | small residuals, positive margins, or rank changes |

The first visual is a dependency map. It is intentionally simple: before computing anything, the reader should see how angle conventions, quaternions, chart singularities, and normalization depend on one another. The second visual is the main lab for this appendix. It compares a quaternion scalar component with the Cayley norm along a one-parameter family of rotations, making the quaternion double cover and the Cayley pi barrier visible on the same horizontal angle axis. The third visual is a numeric check that records the margins used later: orthogonality, determinant, residual size, and artifact image variation.

## Working Principles

The most reliable way to learn robotics geometry is to move between three views. The first view is symbolic: equations name maps and constraints. The second view is numerical: a small instance exposes scale, rank, conditioning, and residuals. The third view is visual: geometry becomes a shape, path, ellipsoid, cone, graph, or surface. This notebook keeps all three views close together. If a symbolic statement is correct, the numerical experiment should produce the expected small residual or positive margin. If a visual is meaningful, it should make a specific invariant or failure mode easier to see.

For other representations of rotations, the relevant failure modes are not side details; they are part of the concept. Euler and roll-pitch-yaw singularities do not mean that the rigid body has stopped rotating; they mean the chosen coordinates have stopped being a reliable chart. Quaternion sign flips do not change the attitude, but they can create artificial jumps in a plotted signal or an interpolated trajectory. Cayley parameters do not merely become large near pi; their blow-up is the coordinate system telling us that this rotation lies on the excluded boundary. A robust robotics model names those edges, draws them, and then tests a small case so the reader can recognize the issue in later code.

## Applied Lab

Compare parameter curves as the rotation angle approaches pi.

In the lab, vary the rotation angle while keeping the axis fixed and predict which coordinate should remain bounded. The quaternion scalar should move smoothly from one toward zero as the angle approaches pi, while the Cayley norm should climb sharply because the chart has no finite value at pi. The rotation matrix reconstructed from either representation should still preserve vector lengths before the Cayley chart reaches its boundary. This prediction step is small, but it is what turns a figure into a mathematical instrument.

## Pitfalls To Watch

- A coordinate singularity is not a physical singularity.
- Quaternion sign flips can break interpolation if ignored.

These pitfalls are deliberately operational. If a computation becomes confusing, check frame labels, units, rank, and the distinction between a physical object and its coordinates. Many robotics errors are not arithmetic mistakes; they are mismatches between a representation and the geometry it claims to encode.

## Takeaways

- Rotation representations are engineering tradeoffs.
- The invariant object is the rotation, not the coordinate chart.

By the end of this notebook, the reader should be able to explain the chapter's main object, build a small computed example, interpret the generated visual artifact, and state at least one numerical sanity check that would catch an implementation mistake.

In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np

HERE = Path.cwd().resolve()
BOOK_ROOT = next(
    parent for parent in [HERE, *HERE.parents]
    if (parent / "AGENTS.md").exists() and (parent / "Mordern Robotics.pdf").exists()
)
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, save_json
from utils.validation import assert_artifact, image_stats

print(f"BOOK_ROOT={BOOK_ROOT}")

In [ ]:
CHAPTER = {
  "number": 15,
  "slug": "appendix-b-other-representations-of-rotations",
  "title": "Other Representations of Rotations",
  "notebook": "appendix-b-other-representations-of-rotations.ipynb",
  "printed_start": 575,
  "printed_end": 584,
  "pdf_start": 593,
  "pdf_end": 602,
  "part_slug": "part-05-reference-appendices",
  "part_title": "Reference Appendices",
  "theme": "appendix",
  "visual_focus": "Euler angles, roll-pitch-yaw, quaternions, and Cayley parameters",
  "visual_kind": "rotation parameter comparison curves",
  "artifact_stem": "rotation-representations",
  "inspection_target": "where each parameterization is smooth, singular, or redundant",
  "question": "Why do so many rotation coordinates exist, and what does each one trade away?",
  "terms": [
    "Euler angles",
    "roll-pitch-yaw",
    "unit quaternion",
    "Cayley-Rodrigues",
    "singularity",
    "double cover",
    "minimal coordinate"
  ],
  "translation": [
    "Three-parameter coordinates are compact but cannot be globally nonsingular.",
    "Unit quaternions avoid gimbal lock by adding a constraint and a sign ambiguity.",
    "Cayley parameters exclude rotations by pi."
  ],
  "lab": "Compare parameter curves as the rotation angle approaches pi.",
  "pitfalls": [
    "A coordinate singularity is not a physical singularity.",
    "Quaternion sign flips can break interpolation if ignored."
  ],
  "takeaways": [
    "Rotation representations are engineering tradeoffs.",
    "The invariant object is the rotation, not the coordinate chart."
  ]
}

from utils.visuals import build_storyboard
storyboard = build_storyboard(CHAPTER)
storyboard

In [ ]:
from utils.visuals import build_chapter_visuals

outputs = build_chapter_visuals(CHAPTER)
for artifact in outputs['figures']:
    display_artifact(artifact, width=760)
outputs['metrics']

## Worked Example

The worked example below builds one attitude three ways: from exponential coordinates, from a unit quaternion, and from Cayley-Rodrigues parameters. The point is not to prefer one notation in the abstract. It is to check that each coordinate reconstructs the same rotation matrix, that the matrix still satisfies the SO(3) constraints, and that the quaternion sign ambiguity is harmless only after we compare the represented rotation rather than the raw four-vector.

In [ ]:
from utils.lie import hat_so3, so3_exp


def quat_to_matrix(q):
    q = np.asarray(q, dtype=float)
    q = q / np.linalg.norm(q)
    scalar, vector = q[0], q[1:]
    V = hat_so3(vector)
    return np.eye(3) + 2 * scalar * V + 2 * (V @ V)


def cayley_to_matrix(a):
    A = hat_so3(a)
    return (np.eye(3) + A) @ np.linalg.inv(np.eye(3) - A)


axis = np.array([0.3, -0.5, 0.8])
axis = axis / np.linalg.norm(axis)
angle = 1.35
R_exp = so3_exp(axis * angle)
q = np.r_[math.cos(angle / 2), math.sin(angle / 2) * axis]
R_quat = quat_to_matrix(q)
R_quat_negative = quat_to_matrix(-q)
cayley = math.tan(angle / 2) * axis
R_cayley = cayley_to_matrix(cayley)
probe = np.array([0.7, -0.2, 0.4])

worked_example = {
    "angle_radians": float(angle),
    "quaternion_norm": float(np.linalg.norm(q)),
    "cayley_norm": float(np.linalg.norm(cayley)),
    "quat_matrix_error": float(np.linalg.norm(R_quat - R_exp)),
    "quat_sign_matrix_error": float(np.linalg.norm(R_quat_negative - R_quat)),
    "cayley_matrix_error": float(np.linalg.norm(R_cayley - R_exp)),
    "orthogonality_residual": float(np.linalg.norm(R_exp.T @ R_exp - np.eye(3))),
    "determinant_error": float(abs(np.linalg.det(R_exp) - 1.0)),
    "rotated_probe_difference": float(np.linalg.norm(R_quat @ probe - R_cayley @ probe)),
}
assert worked_example["quat_matrix_error"] < 1e-12
assert worked_example["quat_sign_matrix_error"] < 1e-12
assert worked_example["cayley_matrix_error"] < 1e-12
assert worked_example["orthogonality_residual"] < 1e-12
assert worked_example["determinant_error"] < 1e-12
worked_example

## Applied Lab

The applied lab pushes a fixed-axis rotation toward pi and watches the coordinates respond. This is the appendix B stress test: the physical rotation remains ordinary, the quaternion representation stays bounded, and the Cayley chart reports trouble by growing rapidly. The numerical summary below should agree with the visual lab and should make clear that a coordinate singularity is a chart failure, not a failure of SO(3).

In [ ]:
angles = np.linspace(0.0, 0.985 * np.pi, 80)
fixed_axis = np.array([0.0, 0.0, 1.0])
probe = np.array([1.0, -0.3, 0.2])
quat_scalars = np.cos(angles / 2)
cayley_norms = np.tan(angles / 2)
quat_residuals = []
vector_length_errors = []

for a in angles:
    R = so3_exp(fixed_axis * a)
    q = np.r_[math.cos(a / 2), math.sin(a / 2) * fixed_axis]
    quat_residuals.append(np.linalg.norm(quat_to_matrix(q) - R))
    vector_length_errors.append(abs(np.linalg.norm(R @ probe) - np.linalg.norm(probe)))

lab_summary = {
    "angle_stop_fraction_of_pi": float(angles[-1] / np.pi),
    "quaternion_scalar_near_pi": float(quat_scalars[-1]),
    "cayley_norm_near_pi": float(cayley_norms[-1]),
    "max_quaternion_matrix_residual": float(np.max(quat_residuals)),
    "max_vector_length_error": float(np.max(vector_length_errors)),
}
assert lab_summary["max_quaternion_matrix_residual"] < 1e-12
assert lab_summary["max_vector_length_error"] < 1e-12

lab_summary

## Sanity Checks

The final cell asserts that the generated artifacts exist, are large enough to be useful, and have nontrivial pixel variation. It also stores a JSON summary under the appendix B artifact subtree so the notebook leaves a machine-checkable trace of the representation checks, including the worked example and the near-pi coordinate sweep.

In [ ]:
artifact_stats = {}
for artifact in outputs['figures']:
    resolved = assert_artifact(artifact)
    stats = image_stats(resolved)
    assert stats['pixel_std'] > 2.0, f'low variation image: {resolved}'
    artifact_stats[resolved.name] = stats
assert_artifact(outputs['checks'], min_size=100)
sanity = {'chapter': CHAPTER['slug'], 'metrics': outputs['metrics'], 'worked_example': worked_example, 'lab_summary': lab_summary, 'artifact_stats': artifact_stats}
sanity_path = save_json(sanity, CHAPTER['slug'], 'checks', 'notebook-sanity.json')
display_artifact(sanity_path)
sanity